# spelunking lab on Colab

Run all after each VM start. Rerunning is safe: `up.sh` skips what is already done.

Colab Secrets (key icon, left bar), notebook access on:
- `GITHUB_TOKEN`: fine-grained token, this repo only, Contents read and write
- `VLLM_API_KEY`: any long random string; the laptop needs the same one
- `NGROK_AUTHTOKEN`, `NGROK_DOMAIN` (optional): without them you get a Cloudflare quick tunnel with a new URL each time
- `HF_TOKEN` (optional): only for gated models

In [ ]:
import os, subprocess
from google.colab import userdata

for name in ["GITHUB_TOKEN", "VLLM_API_KEY", "NGROK_AUTHTOKEN", "NGROK_DOMAIN", "HF_TOKEN"]:
    try:
        os.environ[name] = userdata.get(name)
    except userdata.SecretNotFoundError:
        pass

os.environ["REPO_URL"] = "https://github.com/krishmula/spelunking.git"
os.environ["MODEL"] = "Qwen/Qwen3-4B"
os.environ["VLLM_VERSION"] = "0.31.0"
# os.environ["VLLM_GIT"] = "https://github.com/krishmula/vllm.git@lab"   # to run your engine edits

if not os.path.isdir("/content/spelunking"):
    subprocess.run(["git", "-c", "credential.helper=", "-c",
                    "credential.helper=!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f",
                    "clone", os.environ["REPO_URL"], "/content/spelunking"], check=True)
!bash /content/spelunking/colab/up.sh

Laptop: `VLLM_API_KEY=... python colab/check.py <URL>`

Measure inside the VM, never through the tunnel: the tunnel's round trip lands on every TTFT and TPOT.
Experiments run here, e.g. `!cd /content/spelunking/experiments/E02-... && python run_experiment.py`.

In [ ]:
%%bash
# Save every run folder to a branch before the VM goes away. Merge it on the laptop.
cd /content/spelunking
branch=colab-runs/$(date +%Y%m%d-%H%M%S)
git switch -c "$branch"
git add experiments/*/runs
git -c user.name=colab -c user.email=colab@localhost commit -m "Runs from Colab $(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)"
git -c credential.helper= -c credential.helper='!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f' push -u origin "$branch"
git switch -

In [ ]:
# Keeps the session from idling out while you work from the laptop. Stop it to run other cells.
!tail -f /content/state/server.log